# φ Redundancy vs Binary: Block-Structured Computational Sparsity

This notebook tests whether golden-ratio (`φ`) redundancy can create **structured zero blocks** in neural-network digit planes, and compares the result with a conventional binary/INT8 representation.

We separate two questions:

### Experiment A — Exact representation
The already-quantized numerical weight is held fixed.

- **φ:** may choose any equivalent 8-digit representation of the same φ value.
- **Binary:** ordinary binary magnitude representation is unique, so there is no analogous exact rewrite freedom.

We ask whether φ redundancy can create additional all-zero blocks without changing the quantized matrix.

### Experiment B — Matched-distortion structured quantization
Both φ and binary are allowed to change quantized values. We optimize locally for structured block sparsity while tracking:

- reconstruction MSE / MAE,
- MNIST accuracy,
- active digits,
- active digit-plane blocks,
- zero-block fraction.

This is the more practically relevant comparison.

> Zero blocks and active digits are structural compute proxies. They do **not** prove wall-clock speedup on existing CPUs/GPUs.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math, itertools
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)


## 1. MNIST and baseline model


In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    loss_sum = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}: loss={loss_sum/len(train_loader):.4f}")


In [ ]:
def evaluate(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            pred = model(images).argmax(1)
            total += labels.numel()
            correct += (pred == labels).sum().item()
    return correct / total

def eval_weights(new_W, original_W):
    with torch.no_grad():
        model.fc1.weight.copy_(new_W)
    acc = evaluate(model)
    with torch.no_grad():
        model.fc1.weight.copy_(original_W)
    return acc

def metrics(W, W_hat):
    mse = torch.mean((W-W_hat)**2).item()
    mae = torch.mean(torch.abs(W-W_hat)).item()
    return mse, mae

W = model.fc1.weight.detach().clone()
W_original = W.clone()
fp32_acc = evaluate(model)

print("Shape:", tuple(W.shape))
print("Weights:", W.numel())
print(f"FP32 accuracy: {fp32_acc*100:.2f}%")


## 2. Conventional INT8 / binary baseline

We use symmetric INT8:

\[
q=\mathrm{clip}(\mathrm{round}(w/s),-127,127)
\]

with sign plus a 7-bit magnitude decomposition.

The binary digit planes correspond to magnitude bits \(1,2,4,\dots,64\).


In [ ]:
qmax = 127
int8_scale = W.abs().max() / qmax
Q8 = torch.clamp(torch.round(W/int8_scale), -qmax, qmax).to(torch.int64)
W_int8 = Q8.float() * int8_scale

int8_mse, int8_mae = metrics(W, W_int8)
int8_acc = eval_weights(W_int8, W_original)

def binary_bits_from_codes(Q, nbits=7):
    mag = Q.abs().flatten().cpu().numpy().astype(np.int64)
    return np.stack([((mag >> k) & 1) for k in range(nbits)], axis=1).astype(np.int8)

binary_bits = binary_bits_from_codes(Q8, 7)

print("INT8 MSE:", int8_mse)
print("INT8 MAE:", int8_mae)
print(f"INT8 accuracy: {int8_acc*100:.2f}%")
print("Binary avg active magnitude bits:", binary_bits.sum(1).mean())


## 3. Build the 8-digit φ codebook and all equivalent representations


In [ ]:
phi = (1 + math.sqrt(5))/2
K = 8
phi_powers = np.array([phi**(-k) for k in range(1,K+1)], dtype=np.float64)

groups = defaultdict(list)
for bits in itertools.product([0,1], repeat=K):
    value = float(np.dot(np.array(bits), phi_powers))
    groups[round(value,10)].append(tuple(bits))

phi_keys = list(groups.keys())
phi_values = np.array(phi_keys, dtype=np.float64)

canonical = {
    key: min(reps, key=lambda b:(sum(b),b))
    for key,reps in groups.items()
}

print("Raw φ patterns:", 2**K)
print("Unique φ magnitudes:", len(groups))
print("Redundant patterns:", 2**K-len(groups))


## 4. Quantize once to φ and freeze the numerical values


In [ ]:
phi_scale = W.abs().max().item() / phi_values.max()
flatW = W.flatten().cpu().numpy()

selected_keys = []
W_phi_flat = np.zeros_like(flatW, dtype=np.float32)

for i,w in enumerate(flatW):
    target = abs(float(w))/phi_scale
    idx = int(np.argmin(np.abs(phi_values-target)))
    key = phi_keys[idx]
    selected_keys.append(key)
    W_phi_flat[i] = (-1 if w < 0 else 1) * phi_scale * key

W_phi = torch.tensor(W_phi_flat.reshape(W.shape), dtype=W.dtype)
phi_mse, phi_mae = metrics(W, W_phi)
phi_acc = eval_weights(W_phi, W_original)

phi_local_bits = np.array([canonical[k] for k in selected_keys], dtype=np.int8)

print("φ MSE:", phi_mse)
print("φ MAE:", phi_mae)
print(f"φ accuracy: {phi_acc*100:.2f}%")
print("φ avg active digits:", phi_local_bits.sum(1).mean())


# Experiment A: Exact block optimization

For each spatial block of the \(128\times784\) weight matrix, we independently search all subsets of φ planes.

For a candidate allowed-plane subset, **every φ value in that block must still be represented exactly**. Among feasible subsets, we choose a solution minimizing

\[
C = \lambda_B(\text{active blocks}) + \text{active digits}.
\]

Because each spatial block is optimized separately, a plane may be zero in one block even if it is required elsewhere.

For binary INT8, exact magnitude representation is unique, so its existing zero blocks form the exact baseline.


In [ ]:
def block_slices(shape, br, bc):
    R,C = shape
    out = []
    for r0 in range(0,R,br):
        for c0 in range(0,C,bc):
            out.append((slice(r0,min(r0+br,R)), slice(c0,min(c0+bc,C))))
    return out

def reshape_bits_to_planes(bits, shape):
    R,C = shape
    return np.stack([bits[:,k].reshape(R,C) for k in range(bits.shape[1])], axis=0)

def block_stats(bits, shape, block_size):
    planes = reshape_bits_to_planes(bits, shape)
    slices = block_slices(shape, block_size, block_size)
    active = 0
    total = planes.shape[0] * len(slices)

    for p in range(planes.shape[0]):
        for rs,cs in slices:
            if np.any(planes[p,rs,cs]):
                active += 1

    return {
        "total_blocks": total,
        "active_blocks": active,
        "zero_blocks": total-active,
        "zero_fraction": (total-active)/total,
        "active_digits": int(bits.sum()),
        "avg_active_digits": float(bits.sum()/bits.shape[0])
    }


In [ ]:
def exact_optimize_phi_blocks(selected_keys, shape, block_size, block_penalty=1000.0):
    R,C = shape
    key_matrix = np.array(selected_keys, dtype=object).reshape(R,C)
    result = np.zeros((R,C,K), dtype=np.int8)

    # Cache best representation for (key, mask).
    cache = {}

    def best_rep(key, mask):
        ck = (key, mask)
        if ck in cache:
            return cache[ck]
        allowed = [(mask >> k) & 1 for k in range(K)]
        candidates = [
            b for b in groups[key]
            if all((not bit) or allowed[k] for k,bit in enumerate(b))
        ]
        ans = None if not candidates else min(candidates, key=lambda b:(sum(b),b))
        cache[ck] = ans
        return ans

    for rs,cs in block_slices(shape, block_size, block_size):
        keys_block = key_matrix[rs,cs]
        unique_keys = set(keys_block.flatten())

        best_cost = float("inf")
        best_map = None

        for mask in range(1, 2**K):
            reps = {}
            feasible = True
            for key in unique_keys:
                rep = best_rep(key, mask)
                if rep is None:
                    feasible = False
                    break
                reps[key] = rep
            if not feasible:
                continue

            # Count only planes actually used by this block.
            temp = np.array([reps[k] for k in keys_block.flatten()], dtype=np.int8)
            actual_active_planes = np.count_nonzero(temp.sum(axis=0))
            digit_count = temp.sum()
            cost = block_penalty*actual_active_planes + digit_count

            if cost < best_cost:
                best_cost = cost
                best_map = reps

        for i in range(keys_block.shape[0]):
            for j in range(keys_block.shape[1]):
                result[rs.start+i, cs.start+j] = best_map[keys_block[i,j]]

    return result.reshape(-1,K)


## 5. Run exact block optimization at several block sizes

Smaller blocks give the optimizer more local freedom. We compare against the exact binary INT8 planes at the same spatial block sizes.


In [ ]:
block_sizes = [4, 8, 16, 32]
exact_results = {}

for bs in block_sizes:
    print("Optimizing exact φ blocks:", bs, "x", bs)

    phi_exact_bits = exact_optimize_phi_blocks(
        selected_keys,
        W.shape,
        bs,
        block_penalty=1000.0
    )

    # Verify every φ numerical magnitude is unchanged.
    decoded = phi_exact_bits.astype(np.float64) @ phi_powers
    assert np.allclose(decoded, np.array(selected_keys), atol=1e-8)

    phi_stats = block_stats(phi_exact_bits, W.shape, bs)
    bin_stats = block_stats(binary_bits, W.shape, bs)

    exact_results[bs] = {
        "phi_bits": phi_exact_bits,
        "phi": phi_stats,
        "binary": bin_stats
    }

    print(
        f"  φ zero blocks: {phi_stats['zero_fraction']*100:.2f}% | "
        f"binary zero blocks: {bin_stats['zero_fraction']*100:.2f}%"
    )


In [ ]:
print(f"{'Block':>8} {'φ zero %':>12} {'Binary zero %':>15} {'φ active digits':>16} {'Binary active bits':>19}")
for bs in block_sizes:
    p = exact_results[bs]["phi"]
    b = exact_results[bs]["binary"]
    print(
        f"{str(bs)+'x'+str(bs):>8} "
        f"{100*p['zero_fraction']:>11.2f}% "
        f"{100*b['zero_fraction']:>14.2f}% "
        f"{p['active_digits']:>16} "
        f"{b['active_digits']:>19}"
    )


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(block_sizes, [100*exact_results[b]["phi"]["zero_fraction"] for b in block_sizes], marker="o", label="φ exact rewrite")
plt.plot(block_sizes, [100*exact_results[b]["binary"]["zero_fraction"] for b in block_sizes], marker="o", label="INT8 binary")
plt.xlabel("Block side length")
plt.ylabel("Zero digit-plane blocks (%)")
plt.title("Exact Representation: Structured Zero Blocks")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Experiment B: Matched-distortion structured sparsity

Exact equality may be too restrictive.

Now both methods may choose nearby representable values. We introduce a block-aware cost:

\[
J = \text{squared reconstruction error}
+ \lambda_D(\text{active digits})
+ \lambda_B(\text{newly activated plane-blocks}).
\]

We use a greedy block optimizer. Within each spatial block, weights are processed from largest absolute magnitude to smallest. For each weight, the candidate representation is chosen using reconstruction error plus penalties for active digits and for activating a previously unused digit plane in that block.

This is an exploratory algorithm, not a globally optimal solver.

For φ, candidates are all 256 raw digit patterns (with redundancy).  
For binary, candidates are the 128 nonnegative 7-bit magnitudes, with sign handled separately.

We sweep penalties to obtain a distortion–structure tradeoff for both methods.


In [ ]:
# Candidate φ raw patterns. Redundant patterns are deliberately retained.
phi_candidate_bits = np.array(list(itertools.product([0,1], repeat=K)), dtype=np.int8)
phi_candidate_vals = phi_candidate_bits.astype(np.float64) @ phi_powers

# Binary magnitude candidates 0..127.
bin_candidate_vals = np.arange(128, dtype=np.float64)
bin_candidate_bits = np.stack(
    [[(q >> k) & 1 for k in range(7)] for q in range(128)],
    axis=0
).astype(np.int8)

def greedy_structured_quantize(
    W_np, scale, candidate_vals, candidate_bits,
    block_size, lambda_digit, lambda_block
):
    R,C = W_np.shape
    P = candidate_bits.shape[1]
    out_bits = np.zeros((R,C,P), dtype=np.int8)
    out_vals = np.zeros((R,C), dtype=np.float64)

    for rs,cs in block_slices((R,C), block_size, block_size):
        block = W_np[rs,cs]
        coords = [(i,j) for i in range(block.shape[0]) for j in range(block.shape[1])]
        coords.sort(key=lambda ij: -abs(block[ij]))

        used_planes = np.zeros(P, dtype=bool)

        for i,j in coords:
            w = float(block[i,j])
            target = abs(w)

            reconstructed_mags = scale * candidate_vals
            err = (target - reconstructed_mags)**2

            digit_cost = lambda_digit * candidate_bits.sum(axis=1)

            newly_used = candidate_bits[:, ~used_planes].sum(axis=1)
            block_cost = lambda_block * newly_used

            total_cost = err + digit_cost + block_cost
            idx = int(np.argmin(total_cost))

            bits = candidate_bits[idx]
            out_bits[rs.start+i, cs.start+j] = bits
            used_planes |= bits.astype(bool)

            sign = -1.0 if w < 0 else 1.0
            out_vals[rs.start+i, cs.start+j] = sign * reconstructed_mags[idx]

    return out_vals, out_bits


## 6. Sweep structured-sparsity penalties

We focus on 8×8 and 16×16 blocks. Penalties are expressed relative to the trained weight scale so the same sweep can be applied to both representations.

The sweep includes zero penalty, which recovers ordinary nearest-value quantization using each candidate set.


In [ ]:
W_np = W.cpu().numpy()

structured_block_sizes = [8, 16]

# Error is squared weight error, typically around 1e-6 to 1e-4.
# These values deliberately span weak to aggressive structure penalties.
penalty_multipliers = [
    (0.0, 0.0),
    (0.01, 0.05),
    (0.02, 0.10),
    (0.05, 0.25),
    (0.10, 0.50),
    (0.20, 1.00),
    (0.50, 2.00),
]

# Reference squared scale turns dimensionless multipliers into weight-error units.
ref = float(W.abs().max().item()**2)

structured_results = []

for bs in structured_block_sizes:
    for ld_mult, lb_mult in penalty_multipliers:
        ld = ld_mult * ref
        lb = lb_mult * ref

        print(f"block={bs}, digit_penalty={ld_mult}, block_penalty={lb_mult}")

        # φ
        phi_vals_opt, phi_bits_opt = greedy_structured_quantize(
            W_np, phi_scale,
            phi_candidate_vals, phi_candidate_bits,
            bs, ld, lb
        )
        Wp = torch.tensor(phi_vals_opt, dtype=W.dtype)
        pmse,pmae = metrics(W,Wp)
        pstats = block_stats(phi_bits_opt.reshape(-1,K), W.shape, bs)
        pacc = eval_weights(Wp,W_original)

        structured_results.append({
            "method":"PHI",
            "block_size":bs,
            "digit_mult":ld_mult,
            "block_mult":lb_mult,
            "mse":pmse,
            "mae":pmae,
            "accuracy":pacc,
            **pstats
        })

        # Binary
        bin_vals_opt, bin_bits_opt = greedy_structured_quantize(
            W_np, float(int8_scale),
            bin_candidate_vals, bin_candidate_bits,
            bs, ld, lb
        )
        Wb = torch.tensor(bin_vals_opt, dtype=W.dtype)
        bmse,bmae = metrics(W,Wb)
        bstats = block_stats(bin_bits_opt.reshape(-1,7), W.shape, bs)
        bacc = eval_weights(Wb,W_original)

        structured_results.append({
            "method":"BINARY",
            "block_size":bs,
            "digit_mult":ld_mult,
            "block_mult":lb_mult,
            "mse":bmse,
            "mae":bmae,
            "accuracy":bacc,
            **bstats
        })


## 7. Structured-sparsity result table


In [ ]:
print(
    f"{'Method':<8} {'Block':>6} {'λd':>6} {'λb':>6} "
    f"{'MSE':>12} {'Acc':>8} {'ZeroBlk%':>10} "
    f"{'AvgDigits':>10} {'ActBlocks':>10}"
)

for r in structured_results:
    print(
        f"{r['method']:<8} {r['block_size']:>6} "
        f"{r['digit_mult']:>6.2f} {r['block_mult']:>6.2f} "
        f"{r['mse']:>12.8f} {100*r['accuracy']:>7.2f}% "
        f"{100*r['zero_fraction']:>9.2f}% "
        f"{r['avg_active_digits']:>10.4f} "
        f"{r['active_blocks']:>10}"
    )


## 8. Pareto plot: MSE vs zero-block fraction

A point toward the **upper-left** is attractive: more zero blocks at lower reconstruction error.


In [ ]:
for bs in structured_block_sizes:
    plt.figure(figsize=(8,5))

    for method in ["PHI","BINARY"]:
        rows = [
            r for r in structured_results
            if r["block_size"]==bs and r["method"]==method
        ]
        xs = [r["mse"] for r in rows]
        ys = [100*r["zero_fraction"] for r in rows]
        plt.plot(xs,ys,marker="o",label=method)

        for r,x,y in zip(rows,xs,ys):
            plt.annotate(
                f"{r['block_mult']:.2f}",
                (x,y),
                xytext=(4,4),
                textcoords="offset points",
                fontsize=8
            )

    plt.xscale("log")
    plt.xlabel("Weight reconstruction MSE (log scale)")
    plt.ylabel("Zero digit-plane blocks (%)")
    plt.title(f"{bs}×{bs}: Distortion vs Structured Sparsity")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


## 9. Pareto plot: accuracy vs zero-block fraction

This asks the network-level question directly: how much structured sparsity can each representation obtain before predictive accuracy deteriorates?


In [ ]:
for bs in structured_block_sizes:
    plt.figure(figsize=(8,5))

    for method in ["PHI","BINARY"]:
        rows = [
            r for r in structured_results
            if r["block_size"]==bs and r["method"]==method
        ]
        xs = [100*r["zero_fraction"] for r in rows]
        ys = [100*r["accuracy"] for r in rows]
        plt.plot(xs,ys,marker="o",label=method)

    plt.xlabel("Zero digit-plane blocks (%)")
    plt.ylabel("MNIST accuracy (%)")
    plt.title(f"{bs}×{bs}: Accuracy vs Structured Sparsity")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


## 10. Matched-MSE comparison

For each φ result, find the binary result at the same block size with the closest reconstruction MSE. This is not perfect interpolation, but it gives a simple matched-distortion comparison.


In [ ]:
print(
    f"{'Block':>6} {'φ MSE':>12} {'φ Zero%':>10} "
    f"{'Bin MSE':>12} {'Bin Zero%':>10} {'Δ Zero pp':>10}"
)

for bs in structured_block_sizes:
    phis = [r for r in structured_results if r["block_size"]==bs and r["method"]=="PHI"]
    bins = [r for r in structured_results if r["block_size"]==bs and r["method"]=="BINARY"]

    for p in phis:
        b = min(bins, key=lambda x: abs(np.log10(max(x["mse"],1e-20))-np.log10(max(p["mse"],1e-20))))
        delta = 100*(p["zero_fraction"]-b["zero_fraction"])
        print(
            f"{bs:>6} {p['mse']:>12.8f} {100*p['zero_fraction']:>9.2f}% "
            f"{b['mse']:>12.8f} {100*b['zero_fraction']:>9.2f}% "
            f"{delta:>9.2f}"
        )


# 11. Final summary

The most important interpretations are:

### Exact experiment
If φ has a higher zero-block fraction than binary **without changing any φ quantized value**, that is direct evidence that algebraic redundancy can reorganize computation into more structured sparsity.

### Matched-distortion experiment
At similar MSE or similar accuracy, compare:

- zero-block fraction,
- active blocks,
- average active digits.

A useful φ advantage would require it to achieve **more structured sparsity at comparable distortion/accuracy**, not merely more sparsity because it tolerates much larger error.

### Caveat
INT8 is natively accelerated on modern hardware. Even a structural φ advantage in these metrics does not imply faster execution. A positive result here would justify the next stage: a hardware-aware operation/memory model and eventually a specialized kernel.


In [ ]:
print("="*72)
print("CORE BASELINES")
print("="*72)
print(f"FP32 accuracy: {fp32_acc*100:.2f}%")
print(f"INT8: MSE={int8_mse:.10f}, accuracy={int8_acc*100:.2f}%, avg bits={binary_bits.sum(1).mean():.4f}")
print(f"PHI : MSE={phi_mse:.10f}, accuracy={phi_acc*100:.2f}%, avg digits={phi_local_bits.sum(1).mean():.4f}")

print("\n"+"="*72)
print("EXACT BLOCK REWRITE")
print("="*72)
for bs in block_sizes:
    p=exact_results[bs]["phi"]; b=exact_results[bs]["binary"]
    print(
        f"{bs}x{bs}: "
        f"PHI zero={100*p['zero_fraction']:.2f}% active_digits={p['active_digits']} | "
        f"BINARY zero={100*b['zero_fraction']:.2f}% active_bits={b['active_digits']}"
    )

print("\n"+"="*72)
print("BEST ZERO-BLOCK RESULT WITH ACCURACY >= FP32 - 0.5 percentage points")
print("="*72)

threshold = fp32_acc - 0.005
for bs in structured_block_sizes:
    for method in ["PHI","BINARY"]:
        feasible = [
            r for r in structured_results
            if r["block_size"]==bs
            and r["method"]==method
            and r["accuracy"] >= threshold
        ]
        if feasible:
            best=max(feasible,key=lambda r:r["zero_fraction"])
            print(
                f"{method} {bs}x{bs}: zero={100*best['zero_fraction']:.2f}%, "
                f"MSE={best['mse']:.8f}, acc={100*best['accuracy']:.2f}%, "
                f"avg_digits={best['avg_active_digits']:.4f}, "
                f"λd={best['digit_mult']}, λb={best['block_mult']}"
            )
        else:
            print(f"{method} {bs}x{bs}: no result within accuracy threshold")
